# AAL Sales Analysis — Australia (Q4 2020)

**Course-End Project | Applied Data Science with Python**

## Business context
AAL is an Australian apparel retailer serving customers across different age groups. This project analyzes fourth-quarter 2020 sales to identify high- and low-revenue states, understand customer-group performance, and examine sales activity across different times of the day. The findings are intended to support sales and marketing decisions.

**Dataset:** `AusApparalSales4thQrt2020.csv`  
**Expected dataset size:** 7,560 records × 6 columns  
**Expected columns:** `Date`, `Time`, `State`, `Group`, `Unit`, `Sales`

## Project objectives
- Inspect and clean the dataset, including missing and potentially incorrect values.
- Apply Min-Max normalization to the `Unit` and `Sales` columns.
- Explain and use Pandas `groupby()` for summarizing sales.
- Calculate descriptive statistics for `Sales` and `Unit`.
- Identify the highest- and lowest-performing customer groups.
- Generate weekly, monthly, and quarterly sales reports.
- Compare state-wise and customer-group sales.
- Identify peak and off-peak sales periods.
- Create clear visualizations to support sales and marketing recommendations.

## Libraries
Python, Pandas, NumPy, Matplotlib, Seaborn, and SciPy / scikit-learn.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 2. Load the dataset

In [ ]:
DATA_PATH = "AusApparalSales4thQrt2020.csv"  # Keep the CSV in the same folder as this notebook

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nDataset information:")
df.info()

print("\\nDescriptive overview:")
display(df.describe(include="all").T)

## 3. Data wrangling and quality checks

The problem statement asks for missing and incorrect-entry checks using `isna()` and `notna()`. The next cells inspect these issues before any rows are changed.

In [ ]:
# Missing values in each column
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percent": (df.isna().mean() * 100).round(2),
    "Non_Missing_Count": df.notna().sum()
})
display(missing_summary)

print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Convert fields to appropriate data types.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Unit"] = pd.to_numeric(df["Unit"], errors="coerce")
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")

# Inspect rows containing missing values after conversion.
invalid_rows = df[df[["Date", "Time", "State", "Group", "Unit", "Sales"]].isna().any(axis=1)]
print("Rows with missing or unparseable required values:", len(invalid_rows))
display(invalid_rows.head(10))

In [ ]:
# Check categorical values and numeric ranges.
for column in ["Time", "State", "Group"]:
    print(f"\\nUnique values in {column}:")
    print(sorted(df[column].dropna().astype(str).unique()))

print("\\nNumeric value checks:")
display(df[["Unit", "Sales"]].describe())

print("Rows with Unit <= 0:", (df["Unit"] <= 0).sum())
print("Rows with Sales < 0:", (df["Sales"] < 0).sum())

### 3.1 Data-cleaning recommendations

- Investigate missing dates, states, customer groups, units, or sales values before deciding how to treat them.
- Drop rows only when essential information cannot be reliably recovered and the number of affected rows is small.
- Impute numeric values only when a defensible method exists; document the method and avoid arbitrary replacements.
- Check unusual or negative `Unit` and `Sales` values against the source rather than automatically deleting them.
- Remove duplicate rows only after confirming that they are genuine duplicates.

The following cell creates an analysis copy by removing rows with missing required fields and exact duplicate records. Review the checks above before using this approach on a different dataset.

In [ ]:
required_columns = ["Date", "Time", "State", "Group", "Unit", "Sales"]

sales_df = df.dropna(subset=required_columns).drop_duplicates().copy()

print("Original rows:", len(df))
print("Rows after removing missing required values and exact duplicates:", len(sales_df))
print("Rows removed:", len(df) - len(sales_df))

## 4. Data normalization

Min-Max normalization rescales a numeric feature to a range from 0 to 1. The original `Unit` and `Sales` values are retained; normalized values are stored in separate columns for comparison and downstream use.

In [ ]:
scaler = MinMaxScaler()

sales_df[["Unit_Normalized", "Sales_Normalized"]] = scaler.fit_transform(
    sales_df[["Unit", "Sales"]]
)

display(
    sales_df[
        ["Unit", "Unit_Normalized", "Sales", "Sales_Normalized"]
    ].head()
)

display(
    sales_df[["Unit_Normalized", "Sales_Normalized"]].agg(["min", "max"])
)

## 5. Using GroupBy for data analysis

Pandas `groupby()` splits records into groups based on one or more columns, applies an aggregation such as `sum()`, `mean()`, or `count()`, and combines the results into a summary.

For this project, grouping by `State` and `Group` makes it possible to compare revenue across states and customer demographics. It is useful for summarization and comparison; it does not itself merge unrelated datasets.

In [ ]:
state_sales = (
    sales_df.groupby("State", as_index=False)
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
            .sort_values("Total_Sales", ascending=False)
)

group_sales = (
    sales_df.groupby("Group", as_index=False)
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
            .sort_values("Total_Sales", ascending=False)
)

state_group_sales = (
    sales_df.groupby(["State", "Group"], as_index=False)
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
)

display(state_sales)
display(group_sales)
display(state_group_sales.head(12))

## 6. Descriptive statistical analysis

Calculate mean, median, mode, and standard deviation for the `Sales` and `Unit` columns.

In [ ]:
def descriptive_summary(series):
    modes = series.mode()
    return pd.Series({
        "Mean": series.mean(),
        "Median": series.median(),
        "Mode": modes.iloc[0] if not modes.empty else np.nan,
        "Standard_Deviation": series.std()
    })

descriptive_stats = pd.DataFrame({
    "Sales": descriptive_summary(sales_df["Sales"]),
    "Unit": descriptive_summary(sales_df["Unit"])
}).T

display(descriptive_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(y=sales_df["Sales"], ax=axes[0])
axes[0].set_title("Sales Distribution")
axes[0].set_ylabel("Sales")

sns.boxplot(y=sales_df["Unit"], ax=axes[1])
axes[1].set_title("Unit Distribution")
axes[1].set_ylabel("Units")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=sales_df, x="Sales", kde=True, ax=axes[0])
axes[0].set_title("Distribution of Sales")

sns.histplot(data=sales_df, x="Unit", kde=True, ax=axes[1])
axes[1].set_title("Distribution of Units")

plt.tight_layout()
plt.show()

## 7. Highest- and lowest-performing states and groups

In [ ]:
highest_state = state_sales.iloc[0]
lowest_state = state_sales.iloc[-1]

highest_group = group_sales.iloc[0]
lowest_group = group_sales.iloc[-1]

print(f"Highest-revenue state: {highest_state['State']} — {highest_state['Total_Sales']:,.2f}")
print(f"Lowest-revenue state: {lowest_state['State']} — {lowest_state['Total_Sales']:,.2f}")
print(f"Highest-sales customer group: {highest_group['Group']} — {highest_group['Total_Sales']:,.2f}")
print(f"Lowest-sales customer group: {lowest_group['Group']} — {lowest_group['Total_Sales']:,.2f}")

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=state_sales, x="State", y="Total_Sales")
plt.title("Total Sales by State")
plt.xlabel("State")
plt.ylabel("Total sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=group_sales, x="Group", y="Total_Sales")
plt.title("Total Sales by Customer Group")
plt.xlabel("Customer group")
plt.ylabel("Total sales")
plt.tight_layout()
plt.show()

## 8. State-wise and group-wise sales analysis

Compare each demographic group across states, and each state's contribution across customer groups.

In [ ]:
state_group_pivot = state_group_sales.pivot(
    index="State", columns="Group", values="Total_Sales"
).fillna(0)

display(state_group_pivot)

plt.figure(figsize=(11, 6))
sns.heatmap(state_group_pivot, annot=True, fmt=".0f", cmap="YlGnBu")
plt.title("State-wise Sales by Customer Group")
plt.xlabel("Customer group")
plt.ylabel("State")
plt.tight_layout()
plt.show()

In [ ]:
state_group_sales_plot = state_group_sales.sort_values("Total_Sales", ascending=False)

plt.figure(figsize=(13, 6))
sns.barplot(
    data=state_group_sales_plot,
    x="State",
    y="Total_Sales",
    hue="Group"
)
plt.title("Customer Group Sales Across States")
plt.xlabel("State")
plt.ylabel("Total sales")
plt.xticks(rotation=45)
plt.legend(title="Customer group")
plt.tight_layout()
plt.show()

## 9. Time-of-day analysis

The `Time` field in this dataset represents time-of-day categories. This section compares total sales and units for each available category, then identifies the peak and off-peak periods based on total sales.

In [ ]:
time_sales = (
    sales_df.groupby("Time", as_index=False)
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
            .sort_values("Total_Sales", ascending=False)
)

display(time_sales)

peak_period = time_sales.iloc[0]
off_peak_period = time_sales.iloc[-1]

print(f"Peak sales period: {peak_period['Time']} ({peak_period['Total_Sales']:,.2f})")
print(f"Lowest sales period: {off_peak_period['Time']} ({off_peak_period['Total_Sales']:,.2f})")

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=time_sales, x="Time", y="Total_Sales")
plt.title("Sales by Time of Day")
plt.xlabel("Time period")
plt.ylabel("Total sales")
plt.tight_layout()
plt.show()

## 10. Daily, weekly, monthly, and quarterly reports

The source data covers Q4 2020. The quarterly summary therefore describes the quarter represented in the dataset; it should not be interpreted as a multi-quarter comparison.

In [ ]:
daily_sales = (
    sales_df.groupby("Date", as_index=False)
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
            .sort_values("Date")
)

weekly_sales = (
    sales_df.set_index("Date")
            .resample("W")
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
            .reset_index()
)

monthly_sales = (
    sales_df.set_index("Date")
            .resample("MS")
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
            .reset_index()
)

quarterly_sales = (
    sales_df.set_index("Date")
            .resample("QS")
            .agg(Total_Sales=("Sales", "sum"), Total_Units=("Unit", "sum"))
            .reset_index()
)

print("Daily sales report")
display(daily_sales.head())

print("Weekly sales report")
display(weekly_sales)

print("Monthly sales report")
display(monthly_sales)

print("Quarterly sales report")
display(quarterly_sales)

### 10.1 Daily sales trend

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(data=daily_sales, x="Date", y="Total_Sales")
plt.title("Daily Sales Trend — Q4 2020")
plt.xlabel("Date")
plt.ylabel("Total sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 10.2 Weekly sales trend

In [ ]:
plt.figure(figsize=(13, 5))
sns.lineplot(data=weekly_sales, x="Date", y="Total_Sales", marker="o")
plt.title("Weekly Sales Trend — Q4 2020")
plt.xlabel("Week")
plt.ylabel("Total sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 10.3 Monthly sales trend

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=monthly_sales, x="Date", y="Total_Sales")
plt.title("Monthly Sales — Q4 2020")
plt.xlabel("Month")
plt.ylabel("Total sales")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 10.4 Quarterly sales summary

In [ ]:
display(quarterly_sales)

plt.figure(figsize=(7, 5))
sns.barplot(data=quarterly_sales, x="Date", y="Total_Sales")
plt.title("Quarterly Sales Summary")
plt.xlabel("Quarter")
plt.ylabel("Total sales")
plt.tight_layout()
plt.show()

## 11. Sales dashboard

The following dashboard brings together the main views requested in the project: state and group performance, time-of-day sales, and daily/weekly/monthly/quarterly trends.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

sns.barplot(data=state_sales, x="State", y="Total_Sales", ax=axes[0, 0])
axes[0, 0].set_title("Sales by State")
axes[0, 0].tick_params(axis="x", rotation=40)

sns.barplot(data=group_sales, x="Group", y="Total_Sales", ax=axes[0, 1])
axes[0, 1].set_title("Sales by Customer Group")

sns.barplot(data=time_sales, x="Time", y="Total_Sales", ax=axes[1, 0])
axes[1, 0].set_title("Sales by Time of Day")

sns.lineplot(data=weekly_sales, x="Date", y="Total_Sales", marker="o", ax=axes[1, 1])
axes[1, 1].set_title("Weekly Sales Trend")
axes[1, 1].tick_params(axis="x", rotation=35)

for ax in axes.flat:
    ax.set_xlabel("")
    ax.set_ylabel("Total sales")

plt.suptitle("AAL Sales Dashboard — Q4 2020", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 12. Business insights and recommendations

In [ ]:
# Generate concise, data-driven findings from the calculated summaries.
print("AAL SALES ANALYSIS — KEY FINDINGS")
print("-" * 48)
print(f"Highest-revenue state: {highest_state['State']} ({highest_state['Total_Sales']:,.2f})")
print(f"Lowest-revenue state: {lowest_state['State']} ({lowest_state['Total_Sales']:,.2f})")
print(f"Highest-sales group: {highest_group['Group']} ({highest_group['Total_Sales']:,.2f})")
print(f"Lowest-sales group: {lowest_group['Group']} ({lowest_group['Total_Sales']:,.2f})")
print(f"Peak time period: {peak_period['Time']} ({peak_period['Total_Sales']:,.2f})")
print(f"Lowest time period: {off_peak_period['Time']} ({off_peak_period['Total_Sales']:,.2f})")

### Recommendations for the Sales & Marketing team

Use the results above to tailor recommendations to the actual data:

- **High-revenue states:** Maintain customer engagement and evaluate which customer groups contribute most to revenue.
- **Low-revenue states:** Investigate local customer-group performance and consider targeted promotions, subject to business context and campaign costs.
- **Customer groups:** Adapt product messaging and offers to the groups with the strongest and weakest sales.
- **Time of day:** Align staffing and promotional activity with peak periods; consider targeted offers during lower-activity periods.
- **Reporting cadence:** Monitor daily and weekly trends for short-term changes, and use monthly and quarterly summaries for broader planning.

These are decision-support suggestions. Validate them with operational costs, margins, inventory, and campaign results before implementation.

## 13. Conclusion

This course-end project applies data wrangling, descriptive statistics, Pandas GroupBy operations, Min-Max normalization, data visualization, and business interpretation to AAL's Q4 2020 Australian sales dataset. The analysis is designed to help the Sales & Marketing team compare state performance, understand customer-group contributions, and identify time periods that may benefit from targeted sales programs.

**Project Type:** Course-End Project  
**Course:** Applied Data Science with Python  
**Domain:** Retail Analytics | Data Analysis | Data Visualization